In [ ]:
# 구글 마운트(연동) 해 주세요.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow # 콜랩에서 이미지 출력 지원

video_path = '/content/drive/MyDrive/noooooooooooooo/Colab Notebooks/로키_부트캠프_AI응용/data/greenball.mp4'

In [5]:
def track(image):
    # BGR (numpy array) 입력 >> 초록색 물체 중심 좌표(centroid) 추적

    # 1) 노이즈 제거 >> 가우시안 블러(blur 처리)
    blur = cv2.GaussianBlur(image, (5,5), 0)
    # (5,5) 5*5 filter, 0 :sigmaX (자동계산)

    # 2) BGR >> HSV 색공간 변환
    hsv = cv2.cvtColor(blur, cv2.COLOR_BGR2HSV)
    # hue: 색상(빨강, 초록, 파랑...) / saturation: 채도(색이 진하니?) / value(명도, 빛의 밝기)
    # HSV 로 왜 바꾸나요? 조명이 바뀌어도 색상 찾기가 더 쉬움
    # BGR: 컴퓨터 중심(색 검출이 힘들어요. 파랑, 그린, 빨강 위주임. 조명의 밝기 영향을 많이 받아요)

    # 3) 초록색 범위 설정(필요하면 값 조정 가능)
    # 연한 녹색 - 진한 녹색 범위 설정
    lower_green = np.array([25, 60, 100])    # 색상 40, 채도 70, 명도 70
    upper_green = np.array([50, 255, 255])

    # 4) 마스크(mask) (관심영역 ROI)
    mask = cv2.inRange(hsv, lower_green, upper_green)
    # 초록색 부분만 하얀색(255), 나머지 검은색(0) 만들기
    # 초록색만 오려내는 효과 발생
    # lower green <-> pixel value <-> upper green
    # inRange 이 범위 안에 있으면 >> 255(흰색) / 없으면 0 (검은색)

    # 5) 마스크 다시 블러링 (잡음 noise 제거)
    bmask = cv2.GaussianBlur(mask, (5,5), 0)
    # binary mask 이미지가 0(code0) 또는 255(code 1) 구성된 영상(흑백 영상)

    # 6) 모멘트(moments) 이용하여 중심좌표 계산
    moments = cv2.moments(bmask)
    # key-value {} 저장 (m00: ... 면적(area))
    # m00 : 전체 면적(하얀 픽셀 개수)
    m00 = moments['m00']

    # 중심좌표(centroid) 초기화
    centroid_x, centroid_y = None, None

    # 만약 면적이 0이 아니라면 (면적이 존재한다면)
    if m00 !=0:
       centroid_x = int(moments['m10']/m00) # 한 영역의 중심 x 좌표
       centroid_y = int(moments['m01']/m00) # 한 영역의 중심 y 좌표

       # m10, m01 의미 (1차 모멘트)
       # 1차 모멘트 : 이미지 픽셀 같이 이산 데이터(1, 0 등)에서 각 요소의 값(강도) 위치 곱한 것들의 합
       # 이미지 처리에서 1차 모멘트 계산시 모든 픽셀의 위치(x,y)는 (0,0) (맨 왼쪽 상단) 지점부터 거리
       # m10 : x축 방향에 대한 pixel 강도의 합
       # m01 : y축 방향에 대한 pixel 강도의 합
       # 즉, 해당 좌표의 픽셀의 밝기 의미
       # 참고) 0차 모멘트(m00) 어떤 존재(객체) 있냐, 그것의 크기
       # >> 객체의 질량(면적) >> 모든 픽셀 값을 다 더함

       # 중심좌표(centroid) 공식
       # cx(객체의 가로 중심 좌표) = m10 / m00(전체면적)
       # cy(객체의 세로 중심 좌표) = m01 / m00(전체면적)

    # 기본값: 중심 없음
    # >> 초록색을 못 찾으면
    # 왜 -1 을 썼나요? 유효하지 않은 좌표(x좌표, y좌표 >=0)
    # >> 즉, 이미지 화면에 보이는 것 (0,0)~(w,h)
    # center point 없어 >> 객체 못 찾았어
    # >> 객체가 있다면 (중심점 있다면) >> (cx, cy)

    # -1은 이미지 평면에 존재 안하는 숫자([0,255])
    # 좌상단 (0,0)
    crt = (-1, -1)

    # 중심이 계산된 경우
    if centroid_x != None and centroid_y != None:
      crt = (centroid_x, centroid_y)
      # 이미지 위에 중심점 표시(검은색 점)
      cv2.circle(image, crt, 4, (0,0,0), -1)
      # 4: radius(반지름), -1 : 원의 내부를 지정된 색으로 채워라

    cv2_imshow(image) #colab에서 cv2.imshow() 대신 사용

    return crt

# 메인 실행 부분 (동영상 캡처)

if __name__ == '__main__':
    # 프로그램 시작점(이 파일을 직접 실행할 때만 동작)
    capture = cv2.VideoCapture(video_path)

    if not capture.isOpened():
        print('동영상을 열 수 없습니다.')
    else:
        frame_idx = 0

        while True:
            okay, image = capture.read()

            # 비디오가 마지막이면
            if not okay:
                print('영상 끝까지 재생 완료')
                break

            # 초록색 공 추적
            ctr = track(image)
            print(f'Frame {frame_idx}: centroid = {ctr}')

            frame_idx += 1

        capture.release()

Output hidden; open in https://colab.research.google.com to view.